# Neural Machine Translation - Baseline Evaluation

This notebook evaluates a base model (Mistral-7B) on the test set to establish a baseline BLEU score **before fine-tuning**.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU`

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sacrebleu sentencepiece tqdm

# Check GPU
!nvidia-smi

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 2. Upload Test Set

Upload `data/test_set.jsonl` from your local machine.

In [ ]:
from google.colab import files
import os

# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Upload test set
print("Please upload data/test_set.jsonl:")
uploaded = files.upload()

# Move to data folder
for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f"Saved to data/{filename}")

## 3. Load Test Set

In [ ]:
import json

def load_test_set(path, num_samples=None):
    samples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            samples.append(json.loads(line))
            if num_samples and len(samples) >= num_samples:
                break
    return samples

# Load samples (use 100 for quick test, 2000 for full eval)
NUM_SAMPLES = 100  # Change to 2000 for full evaluation
test_samples = load_test_set('data/test_set.jsonl', NUM_SAMPLES)
print(f"Loaded {len(test_samples)} samples")
print(f"\nSample: {test_samples[0]}")

## 4. Load Base Model (Mistral-7B)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")

## 5. Run Inference

In [ ]:
from tqdm import tqdm

def generate_translation(instruction, input_text):
    """Generate translation using Mistral."""
    prompt = f"[INST] {instruction}\n\n{input_text} [/INST]"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# Run inference
predictions = []
references = []

for sample in tqdm(test_samples, desc="Generating translations"):
    pred = generate_translation(sample["instruction"], sample["input"])
    predictions.append(pred)
    references.append(sample["output"])

print(f"\nGenerated {len(predictions)} translations")

## 6. Calculate BLEU Score

In [ ]:
import sacrebleu

# Calculate BLEU
bleu = sacrebleu.corpus_bleu(predictions, [references])

print("=" * 50)
print("BASELINE BLEU RESULTS")
print("=" * 50)
print(f"BLEU Score: {bleu.score:.2f}")
print(f"Precisions: {[f'{p:.1f}' for p in bleu.precisions]}")
print(f"Brevity Penalty: {bleu.bp:.3f}")
print(f"Signature: {bleu}")

## 7. Save Results

In [ ]:
from datetime import datetime

results = {
    "model": "mistral",
    "model_id": MODEL_ID,
    "num_samples": len(test_samples),
    "timestamp": datetime.now().isoformat(),
    "bleu_score": bleu.score,
    "bleu_details": {
        "precisions": bleu.precisions,
        "brevity_penalty": bleu.bp,
        "length_ratio": bleu.sys_len / bleu.ref_len if bleu.ref_len > 0 else 0,
        "signature": str(bleu)
    },
    "sample_predictions": [
        {
            "input": test_samples[i]["input"][:100],
            "reference": references[i][:100],
            "prediction": predictions[i][:100]
        }
        for i in range(min(5, len(predictions)))
    ]
}

# Save to file
with open('results/baseline_bleu_mistral.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Results saved to results/baseline_bleu_mistral.json")

# Download results
files.download('results/baseline_bleu_mistral.json')

## 8. View Sample Predictions

In [ ]:
print("Sample Predictions:")
print("=" * 70)
for i in range(min(5, len(predictions))):
    print(f"\n--- Sample {i+1} ---")
    print(f"Input:      {test_samples[i]['input'][:80]}...")
    print(f"Reference:  {references[i][:80]}...")
    print(f"Prediction: {predictions[i][:80]}...")